In [1]:
# Inventory Analytics
# Step 1: Load and validate inventory inputs

import pandas as pd
import numpy as np

# Load the cleaned supply chain dataset
file_path = "cleaned_supply_chain_dataset.csv"

supply_chain_df = pd.read_csv(file_path)

# Convert date column
supply_chain_df["date"] = pd.to_datetime(
    supply_chain_df["date"],
    format="%d-%m-%Y",
    errors="raise"
)

# Inventory-related keywords
inventory_keywords = [
    "inventory",
    "stock",
    "lead_time",
    "reorder",
    "order_quantity",
    "unit_cost",
    "unit_price"
]

# Identify inventory-related columns
inventory_cols = [
    col for col in supply_chain_df.columns
    if any(keyword in col.lower() for keyword in inventory_keywords)
]

# Display validation results
print(f"Dataset shape: {supply_chain_df.shape}")

print("\nInventory-related columns found:")
for col in inventory_cols:
    print(f"  - {col}")

print("\nData types:")
print(supply_chain_df[inventory_cols].dtypes)

print("\nMissing values:")
print(supply_chain_df[inventory_cols].isnull().sum())

Dataset shape: (91250, 34)

Inventory-related columns found:
  - inventory_level
  - supplier_lead_time_days
  - reorder_point
  - order_quantity
  - unit_cost
  - unit_price
  - stockout_flag
  - lead_time_demand
  - reorder_point_gap
  - inventory_value
  - inventory_coverage_days
  - inventory_reorder_gap
  - inventory_sales_value
  - reorder_status

Data types:
inventory_level              int64
supplier_lead_time_days      int64
reorder_point                int64
order_quantity               int64
unit_cost                  float64
unit_price                 float64
stockout_flag                int64
lead_time_demand             int64
reorder_point_gap            int64
inventory_value            float64
inventory_coverage_days    float64
inventory_reorder_gap        int64
inventory_sales_value      float64
reorder_status              object
dtype: object

Missing values:
inventory_level              0
supplier_lead_time_days      0
reorder_point                0
order_quantity    

In [2]:
# Step 2A: Inventory Health

inventory_analysis = (
    supply_chain_df
    .groupby(["sku_id", "warehouse_id"])
    .agg(
        average_inventory=("inventory_level", "mean"),
        inventory_value=("inventory_value", "mean"),
        total_units_sold=("units_sold", "sum"),
        average_daily_demand=("units_sold", "mean"),
        demand_std=("units_sold", "std"),
        average_lead_time=("supplier_lead_time_days", "mean"),
        unit_cost=("unit_cost", "mean"),
        unit_price=("unit_price", "mean"),
        existing_reorder_point=("reorder_point", "mean"),
        existing_order_quantity=("order_quantity", "mean"),
        stockout_days=("stockout_flag", "sum")
    )
    .reset_index()
)

# Days of Inventory
inventory_analysis["days_of_inventory"] = (
    inventory_analysis["average_inventory"]
    / inventory_analysis["average_daily_demand"]
)

# Inventory Turnover
inventory_analysis["inventory_turnover"] = (
    inventory_analysis["total_units_sold"]
    / inventory_analysis["average_inventory"]
)

# Demand-to-Inventory Ratio
inventory_analysis["demand_to_inventory_ratio"] = (
    inventory_analysis["average_daily_demand"]
    / inventory_analysis["average_inventory"]
)

# Check the result
print(f"Inventory analysis rows: {len(inventory_analysis):,}")
print(f"Columns created: {len(inventory_analysis.columns)}")

display(inventory_analysis.head(10))

Inventory analysis rows: 250
Columns created: 16


,sku_id,warehouse_id,average_inventory,inventory_value,total_units_sold,average_daily_demand,demand_std,average_lead_time,unit_cost,unit_price,existing_reorder_point,existing_order_quantity,stockout_days,days_of_inventory,inventory_turnover,demand_to_inventory_ratio
0,SKU_1,WH_1,555.547945,7749.893836,7472,20.471233,8.926893,14.0,13.95,20.48,379.0,20.531507,0,27.137982,13.449784,0.036849
1,SKU_1,WH_2,492.498630,5644.034301,7299,19.997260,8.908725,8.0,11.46,19.92,322.0,18.627397,0,24.628305,14.820346,0.040604
2,SKU_1,WH_3,483.684932,2931.130685,7361,20.167123,9.115981,6.0,6.06,8.41,314.0,19.931507,0,23.983834,15.218584,0.041695
3,SKU_1,WH_4,494.764384,3428.717178,7449,20.408219,8.799378,9.0,6.93,10.70,333.0,20.145205,0,24.243388,15.055651,0.041248
4,SKU_1,WH_5,546.912329,10615.568301,7445,20.397260,9.013024,4.0,19.41,23.81,366.0,20.150685,0,26.813029,13.612785,0.037295
5,SKU_10,WH_1,527.383562,6681.949726,7307,20.019178,9.155384,13.0,12.67,18.73,361.0,19.043836,0,26.343917,13.855191,0.037959
6,SKU_10,WH_2,470.391781,6505.518329,7220,19.780822,9.005869,10.0,13.83,22.36,310.0,19.526027,0,23.780194,15.348908,0.042052
7,SKU_10,WH_3,385.254795,5270.285589,7279,19.942466,9.203110,5.0,13.68,17.87,218.0,19.339726,0,19.318313,18.893989,0.051764
8,SKU_10,WH_4,504.126027,9588.477041,7376,20.208219,9.151679,14.0,19.02,24.90,328.0,19.115068,0,24.946584,14.631262,0.040086
9,SKU_10,WH_5,408.734247,3952.460164,7397,20.265753,8.847801,9.0,9.67,15.35,246.0,19.789041,0,20.168717,18.097334,0.049582


In [3]:
# Step 2B: ABC Analysis

# Calculate consumption value
inventory_analysis["consumption_value"] = (
    inventory_analysis["total_units_sold"]
    * inventory_analysis["unit_cost"]
)

# Rank SKU-Warehouse combinations by consumption value
inventory_analysis = inventory_analysis.sort_values(
    "consumption_value",
    ascending=False
).reset_index(drop=True)

# Calculate cumulative consumption value percentage
inventory_analysis["cumulative_value_pct"] = (
    inventory_analysis["consumption_value"].cumsum()
    / inventory_analysis["consumption_value"].sum()
    * 100
)

# Assign ABC classification
inventory_analysis["abc_class"] = np.select(
    [
        inventory_analysis["cumulative_value_pct"] <= 80,
        inventory_analysis["cumulative_value_pct"] <= 95
    ],
    [
        "A",
        "B"
    ],
    default="C"
)

# Validate ABC results
print("ABC Classification Summary:")
print(inventory_analysis["abc_class"].value_counts().sort_index())

print("\nTop 10 SKU-Warehouse combinations by consumption value:")
display(
    inventory_analysis[
        [
            "sku_id",
            "warehouse_id",
            "total_units_sold",
            "unit_cost",
            "consumption_value",
            "cumulative_value_pct",
            "abc_class"
        ]
    ].head(10)
)

ABC Classification Summary:
abc_class
A    163
B     59
C     28
Name: count, dtype: int64

Top 10 SKU-Warehouse combinations by consumption value:


,sku_id,warehouse_id,total_units_sold,unit_cost,consumption_value,cumulative_value_pct,abc_class
0,SKU_38,WH_3,7443,19.61,145957.23,0.653399,A
1,SKU_13,WH_1,7464,19.52,145697.28,1.305635,A
2,SKU_23,WH_5,7459,19.43,144928.37,1.954428,A
3,SKU_31,WH_4,7362,19.66,144736.92,2.602365,A
4,SKU_1,WH_5,7445,19.41,144507.45,3.249274,A
5,SKU_2,WH_1,7310,19.76,144445.60,3.895906,A
6,SKU_24,WH_2,7307,19.69,143874.83,4.539983,A
7,SKU_4,WH_2,7406,19.25,142565.50,5.178199,A
8,SKU_11,WH_1,7284,19.54,142329.36,5.815358,A
9,SKU_32,WH_5,7309,19.44,142086.96,6.451431,A


In [4]:
# Step 2C: XYZ Analysis

# Calculate demand variability
inventory_analysis["demand_cv"] = (
    inventory_analysis["demand_std"]
    / inventory_analysis["average_daily_demand"]
)

# Check the CV distribution before classification
print("Demand CV summary:")
display(
    inventory_analysis["demand_cv"].describe()
)

# Assign XYZ classification
inventory_analysis["xyz_class"] = np.select(
    [
        inventory_analysis["demand_cv"] <= 0.50,
        inventory_analysis["demand_cv"] <= 1.00
    ],
    [
        "X",
        "Y"
    ],
    default="Z"
)

# Show XYZ distribution
print("\nXYZ Classification Summary:")
print(
    inventory_analysis["xyz_class"]
    .value_counts()
    .sort_index()
)

# Show sample results
print("\nSample XYZ results:")
display(
    inventory_analysis[
        [
            "sku_id",
            "warehouse_id",
            "average_daily_demand",
            "demand_std",
            "demand_cv",
            "xyz_class"
        ]
    ].head(10)
)

Demand CV summary:


count    250.000000
mean       0.452470
std        0.014315
min        0.417689
25%        0.441928
50%        0.452184
75%        0.462346
max        0.487385
Name: demand_cv, dtype: float64


XYZ Classification Summary:
xyz_class
X    250
Name: count, dtype: int64

Sample XYZ results:


,sku_id,warehouse_id,average_daily_demand,demand_std,demand_cv,xyz_class
0,SKU_38,WH_3,20.391781,9.807904,0.480973,X
1,SKU_13,WH_1,20.449315,8.955370,0.437930,X
2,SKU_23,WH_5,20.435616,8.616005,0.421617,X
3,SKU_31,WH_4,20.169863,8.451131,0.418998,X
4,SKU_1,WH_5,20.397260,9.013024,0.441874,X
5,SKU_2,WH_1,20.027397,9.259644,0.462349,X
6,SKU_24,WH_2,20.019178,8.761941,0.437677,X
7,SKU_4,WH_2,20.290411,8.956736,0.441427,X
8,SKU_11,WH_1,19.956164,9.105200,0.456260,X
9,SKU_32,WH_5,20.024658,8.705306,0.434729,X


In [5]:
# Step 2D: Combine ABC + XYZ Classification

# Combine ABC and XYZ classes
inventory_analysis["abc_xyz"] = (
    inventory_analysis["abc_class"]
    + inventory_analysis["xyz_class"]
)

# Display ABC-XYZ distribution
print("ABC-XYZ Classification Summary:")
print(
    inventory_analysis["abc_xyz"]
    .value_counts()
    .sort_index()
)

# Display sample results
print("\nSample ABC-XYZ results:")
display(
    inventory_analysis[
        [
            "sku_id",
            "warehouse_id",
            "abc_class",
            "xyz_class",
            "abc_xyz",
            "consumption_value",
            "demand_cv"
        ]
    ].head(10)
)

ABC-XYZ Classification Summary:
abc_xyz
AX    163
BX     59
CX     28
Name: count, dtype: int64

Sample ABC-XYZ results:


,sku_id,warehouse_id,abc_class,xyz_class,abc_xyz,consumption_value,demand_cv
0,SKU_38,WH_3,A,X,AX,145957.23,0.480973
1,SKU_13,WH_1,A,X,AX,145697.28,0.437930
2,SKU_23,WH_5,A,X,AX,144928.37,0.421617
3,SKU_31,WH_4,A,X,AX,144736.92,0.418998
4,SKU_1,WH_5,A,X,AX,144507.45,0.441874
5,SKU_2,WH_1,A,X,AX,144445.60,0.462349
6,SKU_24,WH_2,A,X,AX,143874.83,0.437677
7,SKU_4,WH_2,A,X,AX,142565.50,0.441427
8,SKU_11,WH_1,A,X,AX,142329.36,0.456260
9,SKU_32,WH_5,A,X,AX,142086.96,0.434729


In [6]:
# Step 3: Inventory Risk Classification

# -----------------------------
# 1. Excess Inventory
# -----------------------------

inventory_analysis["excess_inventory_flag"] = (
    inventory_analysis["days_of_inventory"] > 30
)

# -----------------------------
# 2. Stockout Risk
# -----------------------------

inventory_analysis["stockout_risk_flag"] = (
    (inventory_analysis["stockout_days"] > 0)
    |
    (
        inventory_analysis["average_inventory"]
        < inventory_analysis["existing_reorder_point"]
    )
)

# -----------------------------
# 3. Slow Movers
# -----------------------------

inventory_analysis["slow_mover_flag"] = (
    (inventory_analysis["average_daily_demand"] < 20)
    &
    (inventory_analysis["days_of_inventory"] > 30)
)

# -----------------------------
# 4. Fast Movers
# -----------------------------

inventory_analysis["fast_mover_flag"] = (
    (inventory_analysis["average_daily_demand"] >= 20)
    &
    (inventory_analysis["days_of_inventory"] <= 30)
)

# -----------------------------
# 5. At-Risk SKU-Warehouse
# -----------------------------

inventory_analysis["at_risk_flag"] = (
    inventory_analysis["stockout_risk_flag"]
    |
    (
        (inventory_analysis["average_daily_demand"] >= 20)
        &
        (
            inventory_analysis["average_inventory"]
            < inventory_analysis["existing_reorder_point"]
        )
    )
)

# -----------------------------
# 6. Risk Summary
# -----------------------------

print("Inventory Risk Summary:\n")

print(
    "Excess Inventory:",
    inventory_analysis["excess_inventory_flag"].sum()
)

print(
    "Stockout Risk:",
    inventory_analysis["stockout_risk_flag"].sum()
)

print(
    "Slow Movers:",
    inventory_analysis["slow_mover_flag"].sum()
)

print(
    "Fast Movers:",
    inventory_analysis["fast_mover_flag"].sum()
)

print(
    "At-Risk:",
    inventory_analysis["at_risk_flag"].sum()
)

# -----------------------------
# 7. Show At-Risk Items
# -----------------------------

print("\nAt-Risk SKU-Warehouse combinations:")

display(
    inventory_analysis[
        inventory_analysis["at_risk_flag"]
    ][
        [
            "sku_id",
            "warehouse_id",
            "average_inventory",
            "average_daily_demand",
            "days_of_inventory",
            "existing_reorder_point",
            "stockout_days",
            "abc_xyz"
        ]
    ].head(20)
)

Inventory Risk Summary:

Excess Inventory: 0
Stockout Risk: 0
Slow Movers: 0
Fast Movers: 146
At-Risk: 0

At-Risk SKU-Warehouse combinations:


,sku_id,warehouse_id,average_inventory,average_daily_demand,days_of_inventory,existing_reorder_point,stockout_days,abc_xyz


In [7]:
# Step 4: Calculate Demand During Lead Time, Safety Stock and Optimized ROP

# -----------------------------------
# 1. Reload the completed 30-day forecast
# -----------------------------------

forecast_output = pd.read_excel(
    "30_Day_SKU_Warehouse_Forecast.xlsx",
    sheet_name="30_Day_Forecast"
)

print(f"Forecast rows loaded: {len(forecast_output):,}")

# -----------------------------------
# 2. Calculate average forecast demand
# -----------------------------------

forecast_daily = (
    forecast_output
    .groupby(["sku_id", "warehouse_id"])
    .agg(
        forecast_daily_demand=("forecast_demand", "mean")
    )
    .reset_index()
)

# -----------------------------------
# 3. Remove old forecast column if it already exists
# -----------------------------------

if "forecast_daily_demand" in inventory_analysis.columns:
    inventory_analysis = inventory_analysis.drop(
        columns=["forecast_daily_demand"]
    )

# -----------------------------------
# 4. Add forecast demand to inventory analysis
# -----------------------------------

inventory_analysis = inventory_analysis.merge(
    forecast_daily,
    on=["sku_id", "warehouse_id"],
    how="left",
    validate="one_to_one"
)

# -----------------------------------
# 5. Service level assumption
# -----------------------------------

service_level = 0.95
z_score = 1.645

# -----------------------------------
# 6. Demand during lead time
# -----------------------------------

inventory_analysis["forecast_lead_time_demand"] = (
    inventory_analysis["forecast_daily_demand"]
    * inventory_analysis["average_lead_time"]
)

# -----------------------------------
# 7. Safety Stock
# -----------------------------------

inventory_analysis["safety_stock"] = (
    z_score
    * inventory_analysis["demand_std"]
    * np.sqrt(inventory_analysis["average_lead_time"])
)

# -----------------------------------
# 8. Optimized Reorder Point
# -----------------------------------

inventory_analysis["optimized_reorder_point"] = (
    inventory_analysis["forecast_lead_time_demand"]
    + inventory_analysis["safety_stock"]
)

# -----------------------------------
# 9. Compare Optimized ROP vs Existing ROP
# -----------------------------------

inventory_analysis["reorder_point_gap"] = (
    inventory_analysis["optimized_reorder_point"]
    - inventory_analysis["existing_reorder_point"]
)

inventory_analysis["reorder_point_gap_pct"] = (
    inventory_analysis["reorder_point_gap"]
    / inventory_analysis["existing_reorder_point"]
    * 100
)

# -----------------------------------
# 10. Validation
# -----------------------------------

print("\nSafety Stock & Reorder Point Summary")

print(f"Service Level: {service_level:.0%}")
print(f"Z-Score: {z_score}")

print(
    "\nAverage Safety Stock:",
    round(inventory_analysis["safety_stock"].mean(), 2)
)

print(
    "Average Forecast Lead-Time Demand:",
    round(inventory_analysis["forecast_lead_time_demand"].mean(), 2)
)

print(
    "Average Optimized ROP:",
    round(inventory_analysis["optimized_reorder_point"].mean(), 2)
)

print(
    "Average Existing ROP:",
    round(inventory_analysis["existing_reorder_point"].mean(), 2)
)

print("\nSample Safety Stock & ROP results:")

display(
    inventory_analysis[
        [
            "sku_id",
            "warehouse_id",
            "forecast_daily_demand",
            "average_lead_time",
            "demand_std",
            "forecast_lead_time_demand",
            "safety_stock",
            "existing_reorder_point",
            "optimized_reorder_point",
            "reorder_point_gap",
            "reorder_point_gap_pct"
        ]
    ].head(10)
)

Forecast rows loaded: 7,500

Safety Stock & Reorder Point Summary
Service Level: 95%
Z-Score: 1.645

Average Safety Stock: 40.71
Average Forecast Lead-Time Demand: 170.66
Average Optimized ROP: 211.37
Average Existing ROP: 300.07

Sample Safety Stock & ROP results:


,sku_id,warehouse_id,forecast_daily_demand,average_lead_time,demand_std,forecast_lead_time_demand,safety_stock,existing_reorder_point,optimized_reorder_point,reorder_point_gap,reorder_point_gap_pct
0,SKU_38,WH_3,21.527667,8.0,9.807904,172.221333,45.633851,344.0,217.855184,-126.144816,-36.670005
1,SKU_13,WH_1,25.000000,9.0,8.955370,225.000000,44.194753,246.0,269.194753,23.194753,9.428761
2,SKU_23,WH_5,19.767000,3.0,8.616005,59.301000,24.548924,383.0,83.849924,-299.150076,-78.107069
3,SKU_31,WH_4,20.013667,3.0,8.451131,60.041000,24.079162,257.0,84.120162,-172.879838,-67.268419
4,SKU_1,WH_5,23.299667,4.0,9.013024,93.198667,29.652850,366.0,122.851517,-243.148483,-66.434012
5,SKU_2,WH_1,24.370000,13.0,9.259644,316.810000,54.920169,228.0,371.730169,143.730169,63.039548
6,SKU_24,WH_2,20.570000,5.0,8.761941,102.850000,32.229326,365.0,135.079326,-229.920674,-62.991965
7,SKU_4,WH_2,22.860333,13.0,8.956736,297.184333,53.123583,317.0,350.307916,33.307916,10.507229
8,SKU_11,WH_1,16.000000,4.0,9.105200,64.000000,29.956108,286.0,93.956108,-192.043892,-67.148214
9,SKU_32,WH_5,24.091667,9.0,8.705306,216.825000,42.960686,366.0,259.785686,-106.214314,-29.020304


In [8]:
# Step 5: Categorize ROP recommendations

inventory_analysis["rop_recommendation"] = np.select(
    [
        inventory_analysis["reorder_point_gap_pct"] > 10,
        inventory_analysis["reorder_point_gap_pct"] < -10
    ],
    [
        "Increase ROP",
        "Reduce ROP"
    ],
    default="Maintain ROP"
)

print("ROP Recommendation Summary:\n")

display(
    inventory_analysis["rop_recommendation"]
    .value_counts()
    .rename_axis("Recommendation")
    .reset_index(name="SKU_Warehouse_Count")
)

ROP Recommendation Summary:



,Recommendation,SKU_Warehouse_Count
0,Reduce ROP,169
1,Increase ROP,44
2,Maintain ROP,37


In [9]:
# PHASE 8 — Inventory Optimization
# Step 8.1: Identify Excess Inventory using Inventory Coverage

target_inventory_days = 30

inventory_analysis["excess_inventory_units"] = (
    inventory_analysis["average_inventory"]
    - (
        inventory_analysis["forecast_daily_demand"]
        * target_inventory_days
    )
)

inventory_analysis["excess_inventory_flag"] = (
    inventory_analysis["days_of_inventory"]
    > target_inventory_days
)

print("Excess Inventory Summary:\n")

print(
    "Target Inventory Coverage:",
    target_inventory_days,
    "days"
)

print(
    "Potential Excess Inventory:",
    inventory_analysis["excess_inventory_flag"].sum()
)

print(
    "Percentage of SKU-Warehouse combinations:",
    round(
        inventory_analysis["excess_inventory_flag"].mean() * 100,
        2
    ),
    "%"
)

display(
    inventory_analysis[
        inventory_analysis["excess_inventory_flag"]
    ][
        [
            "sku_id",
            "warehouse_id",
            "abc_xyz",
            "average_inventory",
            "forecast_daily_demand",
            "days_of_inventory",
            "excess_inventory_units",
            "inventory_value"
        ]
    ]
    .sort_values("excess_inventory_units", ascending=False)
    .head(10)
)

Excess Inventory Summary:

Target Inventory Coverage: 30 days
Potential Excess Inventory: 0
Percentage of SKU-Warehouse combinations: 0.0 %


,sku_id,warehouse_id,abc_xyz,average_inventory,forecast_daily_demand,days_of_inventory,excess_inventory_units,inventory_value


In [10]:
# Step 8.2: Identify Stockout Risk

inventory_analysis["stockout_risk_flag"] = (
    (inventory_analysis["stockout_days"] > 0)
    |
    (
        inventory_analysis["average_inventory"]
        < inventory_analysis["optimized_reorder_point"]
    )
    |
    (
        inventory_analysis["days_of_inventory"]
        < inventory_analysis["average_lead_time"]
    )
)

print("Stockout Risk Summary:\n")

print(
    "Potential Stockout Risk:",
    inventory_analysis["stockout_risk_flag"].sum()
)

print(
    "Percentage of SKU-Warehouse combinations:",
    round(
        inventory_analysis["stockout_risk_flag"].mean() * 100,
        2
    ),
    "%"
)

display(
    inventory_analysis[
        inventory_analysis["stockout_risk_flag"]
    ][
        [
            "sku_id",
            "warehouse_id",
            "abc_xyz",
            "average_inventory",
            "forecast_daily_demand",
            "days_of_inventory",
            "average_lead_time",
            "optimized_reorder_point",
            "existing_reorder_point",
            "stockout_days"
        ]
    ]
    .sort_values(
        ["days_of_inventory", "average_lead_time"],
        ascending=[True, False]
    )
    .head(10)
)

Stockout Risk Summary:

Potential Stockout Risk: 1
Percentage of SKU-Warehouse combinations: 0.4 %


,sku_id,warehouse_id,abc_xyz,average_inventory,forecast_daily_demand,days_of_inventory,average_lead_time,optimized_reorder_point,existing_reorder_point,stockout_days
29,SKU_50,WH_4,AX,375.230137,22.989333,18.766648,14.0,375.638731,218.0,0


In [11]:
# Step 8.3: Identify Slow Movers

inventory_analysis["slow_mover_flag"] = (
    (inventory_analysis["average_daily_demand"] < 20)
    &
    (inventory_analysis["days_of_inventory"] > 30)
)

print("Slow Mover Summary:\n")

print(
    "Potential Slow Movers:",
    inventory_analysis["slow_mover_flag"].sum()
)

print(
    "Percentage of SKU-Warehouse combinations:",
    round(
        inventory_analysis["slow_mover_flag"].mean() * 100,
        2
    ),
    "%"
)

display(
    inventory_analysis[
        inventory_analysis["slow_mover_flag"]
    ][
        [
            "sku_id",
            "warehouse_id",
            "abc_xyz",
            "average_daily_demand",
            "average_inventory",
            "days_of_inventory",
            "inventory_value",
            "inventory_turnover"
        ]
    ]
    .sort_values("days_of_inventory", ascending=False)
    .head(10)
)

Slow Mover Summary:

Potential Slow Movers: 0
Percentage of SKU-Warehouse combinations: 0.0 %


,sku_id,warehouse_id,abc_xyz,average_daily_demand,average_inventory,days_of_inventory,inventory_value,inventory_turnover


In [12]:

# Step 8.4: Identify Fast Movers

inventory_analysis["fast_mover_flag"] = (
    (inventory_analysis["average_daily_demand"] >= 20)
    &
    (inventory_analysis["days_of_inventory"] <= 30)
)

print("Fast Mover Summary:\n")

print(
    "Potential Fast Movers:",
    inventory_analysis["fast_mover_flag"].sum()
)

print(
    "Percentage of SKU-Warehouse combinations:",
    round(
        inventory_analysis["fast_mover_flag"].mean() * 100,
        2
    ),
    "%"
)

display(
    inventory_analysis[
        inventory_analysis["fast_mover_flag"]
    ][
        [
            "sku_id",
            "warehouse_id",
            "abc_xyz",
            "average_daily_demand",
            "average_inventory",
            "days_of_inventory",
            "optimized_reorder_point",
            "inventory_value",
            "inventory_turnover"
        ]
    ]
    .sort_values(
        "average_daily_demand",
        ascending=False
    )
    .head(10)
)

Fast Mover Summary:

Potential Fast Movers: 146
Percentage of SKU-Warehouse combinations: 58.4 %


,sku_id,warehouse_id,abc_xyz,average_daily_demand,average_inventory,days_of_inventory,optimized_reorder_point,inventory_value,inventory_turnover
165,SKU_33,WH_4,BX,20.720548,434.523288,20.970647,96.522047,4028.030877,17.405281
238,SKU_18,WH_4,CX,20.701370,416.756164,20.131816,138.928014,2229.645479,18.130506
102,SKU_18,WH_5,AX,20.638356,541.405479,26.232975,363.343963,7227.763151,13.913786
143,SKU_34,WH_4,AX,20.616438,430.284932,20.870963,61.740347,4440.540493,17.488412
160,SKU_20,WH_1,AX,20.569863,453.169863,22.030767,265.999694,4327.772192,16.567739
24,SKU_22,WH_3,AX,20.564384,378.416438,18.401545,118.882536,6955.294137,19.835291
201,SKU_29,WH_1,BX,20.561644,412.419178,20.057695,123.411995,2886.934247,18.197505
188,SKU_47,WH_3,BX,20.561644,557.553425,27.116189,302.781262,4343.341178,13.460593
31,SKU_23,WH_3,AX,20.550685,472.578082,22.995734,155.161816,8544.211726,15.872509
44,SKU_13,WH_2,AX,20.517808,519.819178,25.335025,290.850096,8857.718795,14.406933


In [16]:
# Step 8.5: Identify At-Risk SKUs

inventory_analysis["at_risk_flag"] = (
    inventory_analysis["stockout_risk_flag"]
    |
    (
        (inventory_analysis["rop_recommendation"] == "Increase ROP")
        &
        (
            inventory_analysis["fast_mover_flag"]
            |
            inventory_analysis["abc_class"].isin(["A", "B"])
        )
    )
)

inventory_analysis["risk_reason"] = np.select(
    [
        inventory_analysis["stockout_risk_flag"],
        (
            (inventory_analysis["rop_recommendation"] == "Increase ROP")
            &
            inventory_analysis["fast_mover_flag"]
        ),
        (
            (inventory_analysis["rop_recommendation"] == "Increase ROP")
            &
            inventory_analysis["abc_class"].isin(["A", "B"])
        )
    ],
    [
        "Stockout Risk",
        "Fast Mover - Increase ROP",
        "High-Value SKU - Increase ROP"
    ],
    default="No Immediate Risk"
)

print("At-Risk SKU Summary:\n")

print(
    "Potential At-Risk SKU-Warehouse combinations:",
    inventory_analysis["at_risk_flag"].sum()
)

print(
    "Percentage of SKU-Warehouse combinations:",
    round(
        inventory_analysis["at_risk_flag"].mean() * 100,
        2
    ),
    "%"
)

display(
    inventory_analysis[
        inventory_analysis["at_risk_flag"]
    ][
        [
            "sku_id",
            "warehouse_id",
            "abc_xyz",
            "average_daily_demand",
            "average_inventory",
            "days_of_inventory",
            "average_lead_time",
            "existing_reorder_point",
            "optimized_reorder_point",
            "reorder_point_gap_pct",
            "rop_recommendation",
            "risk_reason"
        ]
    ]
    .sort_values(
        "reorder_point_gap_pct",
        ascending=False
    )
    .head(15)
)

At-Risk SKU Summary:

Potential At-Risk SKU-Warehouse combinations: 42
Percentage of SKU-Warehouse combinations: 16.8 %


,sku_id,warehouse_id,abc_xyz,average_daily_demand,average_inventory,days_of_inventory,average_lead_time,existing_reorder_point,optimized_reorder_point,reorder_point_gap_pct,rop_recommendation,risk_reason
29,SKU_50,WH_4,AX,19.994521,375.230137,18.766648,14.0,218.0,375.638731,72.311345,Increase ROP,Stockout Risk
22,SKU_6,WH_2,AX,19.942466,387.200000,19.415854,14.0,217.0,369.017120,70.053973,Increase ROP,High-Value SKU - Increase ROP
5,SKU_2,WH_1,AX,20.027397,416.515068,20.797264,13.0,228.0,371.730169,63.039548,Increase ROP,Fast Mover - Increase ROP
192,SKU_3,WH_1,BX,20.147945,403.580822,20.030868,14.0,240.0,383.492855,59.788689,Increase ROP,Fast Mover - Increase ROP
179,SKU_49,WH_3,BX,20.087671,374.534247,18.644981,12.0,201.0,302.106259,50.301621,Increase ROP,Fast Mover - Increase ROP
77,SKU_36,WH_1,AX,19.854795,450.186301,22.673934,12.0,270.0,398.363917,47.542191,Increase ROP,High-Value SKU - Increase ROP
193,SKU_26,WH_3,BX,19.786301,399.728767,20.202299,14.0,222.0,324.584926,46.209426,Increase ROP,High-Value SKU - Increase ROP
150,SKU_30,WH_2,AX,20.205479,419.084932,20.741153,12.0,231.0,335.555400,45.262078,Increase ROP,Fast Mover - Increase ROP
83,SKU_2,WH_5,AX,20.032877,454.876712,22.706510,14.0,267.0,387.634204,45.181350,Increase ROP,Fast Mover - Increase ROP
129,SKU_43,WH_5,AX,20.194521,415.693151,20.584453,14.0,249.0,354.774549,42.479739,Increase ROP,Fast Mover - Increase ROP


In [15]:
# Step 8.6: Final Phase 8 Summary

phase8_summary = pd.DataFrame({
    "Metric": [
        "Total SKU-Warehouse Combinations",
        "Potential Excess Inventory",
        "Potential Stockout Risk",
        "Potential Slow Movers",
        "Potential Fast Movers",
        "Potential At-Risk Combinations",
        "ROP Increase Recommendations",
        "ROP Reduce Recommendations",
        "ROP Maintain Recommendations"
    ],
    "Count": [
        len(inventory_analysis),
        inventory_analysis["excess_inventory_flag"].sum(),
        inventory_analysis["stockout_risk_flag"].sum(),
        inventory_analysis["slow_mover_flag"].sum(),
        inventory_analysis["fast_mover_flag"].sum(),
        inventory_analysis["at_risk_flag"].sum(),
        (inventory_analysis["rop_recommendation"] == "Increase ROP").sum(),
        (inventory_analysis["rop_recommendation"] == "Reduce ROP").sum(),
        (inventory_analysis["rop_recommendation"] == "Maintain ROP").sum()
    ]
})

phase8_summary["Percentage"] = (
    phase8_summary["Count"]
    / len(inventory_analysis)
    * 100
)

phase8_summary["Percentage"] = (
    phase8_summary["Percentage"].round(1)
)

print("PHASE 8 — INVENTORY OPTIMIZATION SUMMARY\n")

display(phase8_summary)

PHASE 8 — INVENTORY OPTIMIZATION SUMMARY



,Metric,Count,Percentage
0,Total SKU-Warehouse Combinations,250,100.0
1,Potential Excess Inventory,0,0.0
2,Potential Stockout Risk,1,0.4
3,Potential Slow Movers,0,0.0
4,Potential Fast Movers,146,58.4
5,Potential At-Risk Combinations,42,16.8
6,ROP Increase Recommendations,44,17.6
7,ROP Reduce Recommendations,169,67.6
8,ROP Maintain Recommendations,37,14.8
